# Trader 21 - Screening S&P 500 (pełna lista)

**Ulepszona wersja** – screening całego indeksu S&P 500 pod kątem kryteriów Trader 21.

**Kryteria sprawdzane:**
- `trailingPE` < 15
- `priceToBook` < 1.5
- `priceToSalesTrailingTwelveMonths` < 1
- `enterpriseToEbitda` < 10
- `returnOnEquity` > 0.15
- `revenueGrowth` > 0.10

**Co nowego w tej wersji:**
- Zbiera pełne dane (nazwa spółki, wszystkie metryki)
- Zapisuje wyniki do pliku `trader21_sp500_results.csv`
- Ładniejsza tabela + sortowanie
- Obsługuje całą listę S&P 500 (~503 spółki)

Uruchomienie może trwać 10-20 minut.

In [ ]:
!pip install yfinance pandas tqdm -q
import yfinance as yf
import pandas as pd
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

def get_sp500_tickers():
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    tables = pd.read_html(url, header=0)
    return tables[0]['Symbol'].tolist()

def trader21_analiza(ticker):
    try:
        t = yf.Ticker(ticker)
        info = t.info
        
        metrics = {
            'ticker': ticker,
            'name': info.get('longName', 'N/A'),
            'trailingPE': info.get('trailingPE'),
            'priceToBook': info.get('priceToBook'),
            'priceToSalesTrailing12Months': info.get('priceToSalesTrailingTwelveMonths'),
            'enterpriseToEbitda': info.get('enterpriseToEbitda'),
            'returnOnEquity': info.get('returnOnEquity'),
            'revenueGrowth': info.get('revenueGrowth'),
            'currentPrice': info.get('regularMarketPrice') or info.get('currentPrice')
        }
        
        # Liczenie spełnionych kryteriów
        spelnione = 0
        if metrics['trailingPE'] is not None and metrics['trailingPE'] < 15: spelnione += 1
        if metrics['priceToBook'] is not None and metrics['priceToBook'] < 1.5: spelnione += 1
        if metrics['priceToSalesTrailing12Months'] is not None and metrics['priceToSalesTrailing12Months'] < 1: spelnione += 1
        if metrics['enterpriseToEbitda'] is not None and metrics['enterpriseToEbitda'] < 10: spelnione += 1
        if metrics['returnOnEquity'] is not None and metrics['returnOnEquity'] > 0.15: spelnione += 1
        if metrics['revenueGrowth'] is not None and metrics['revenueGrowth'] > 0.10: spelnione += 1
        
        metrics['spelnione'] = spelnione
        return metrics
    except Exception as e:
        return {
            'ticker': ticker,
            'name': f'Error ({str(e)[:30]})',
            'spelnione': 0
        }

print("Pobieram listę S&P 500...")
tickers = get_sp500_tickers()

print(f"Znaleziono {len(tickers)} spółek. Rozpoczynam screening...")
wyniki = []
for tkr in tqdm(tickers, desc="Screening S&P 500"):
    res = trader21_analiza(tkr)
    if res.get('spelnione', 0) >= 3:
        wyniki.append(res)

df = pd.DataFrame(wyniki)
if not df.empty:
    df = df.sort_values('spelnione', ascending=False)
    
    # Wybór kolumn
    cols = ['ticker', 'name', 'spelnione', 'trailingPE', 'priceToBook', 'priceToSalesTrailing12Months', 'enterpriseToEbitda', 'returnOnEquity', 'revenueGrowth', 'currentPrice']
    df = df[cols]
    
    print(f"\n✓ Znaleziono {len(df)} spółek spełniających ≥3 kryteria Trader 21")
    display(df.head(30).round(3).style.set_caption('Top spółki według kryteriów Trader 21'))
    
    # Zapis do CSV
    csv_name = 'trader21_sp500_results.csv'
    df.to_csv(csv_name, index=False)
    print(f"\nWyniki zapisane do pliku: {csv_name}")
    print('Możesz pobrać plik z lewego panelu Colab (ikona folderu)')
else:
    print('Nie znaleziono spółek spełniających kryteria.')